# 03. TGN Final Training

2010–2016 학습, 2017 검증으로 `core` / `others` / `all` 모드 최종 모델을 학습하고 가중치(`best_*_{mode}.pth`)를 저장합니다. 백테스트(07)는 이 가중치를 사용하며, 2018년 이후 데이터는 학습에 쓰이지 않습니다.


In [ ]:
!pip install torch_geometric
!pip install pyinform

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 3.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import glob
import time
import random
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn import Linear

# PyTorch Geometric (TGN 관련 모듈)
from torch_geometric.nn.models.tgn import TGNMemory, IdentityMessage, LastAggregator
from torch_geometric.nn import TransformerConv

# ==========================================
# 0. 시드(Seed) 및 장치(Device) 설정
# ==========================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 장치 설정: {device}")

# ==========================================
# 1. 환경 설정 및 하이퍼파라미터
# ==========================================
memory_dim = 100
time_dim = 100
msg_dim = 2
time_window = 30
rolling_window = 60
threshold = 0.001
epochs = 50
target_col_idx = 0
num_classes = 12

investable_assets = ['gold', 'vnq', 'xlk', 'xlf', 'xlv', 'xle', 'xly']

# 🌟 전체 학습 및 검증 기간을 통으로 설정
train_start = '2010-04-01'
train_end = '2016-12-31'
val_start = '2017-01-01'
val_end = '2017-12-31'

# ==========================================
# 2. 모델 정의 (TGN 연계 모듈)
# ==========================================
def transfer_entropy(source, target, k=1):
    n = len(source)
    if n <= k: return 0.0
    joint_count = sum(1 for i in range(k, n) if source[i-1] == target[i])
    return joint_count / (n - k)

class GraphEmbedding(torch.nn.Module):
    def __init__(self, in_channels, out_channels, msg_dim, time_dim):
        super().__init__()
        self.time_enc = Linear(1, msg_dim)
        self.conv = TransformerConv(
            in_channels=in_channels, out_channels=out_channels // 2,
            heads=2, dropout=0.1, edge_dim=msg_dim
        )
    def forward(self, x, last_update, edge_index, t, msg):
        rel_t = (t - last_update[edge_index[0]]).float()
        rel_t_emb = self.time_enc(rel_t.view(-1, 1))
        edge_attr = msg + rel_t_emb
        return self.conv(x, edge_index, edge_attr)

class NodePredictor(torch.nn.Module):
    def __init__(self, embedding_dim, out_dim=12):
        super().__init__()
        self.lin1 = Linear(embedding_dim, embedding_dim // 2)
        self.lin2 = Linear(embedding_dim // 2, out_dim)

    def forward(self, target_node):
        return self.lin2(torch.relu(self.lin1(target_node)))

# ==========================================
# 3. 데이터 로더 및 전처리 (32개 정상 로드 + 에러 방어)
# ==========================================
def load_csv_data(data_folder='./'):
    csv_files = glob.glob(os.path.join(data_folder, '**', '*.csv'), recursive=True)
    raw_data_dict = {}
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        asset_name = os.path.splitext(file_name)[0]
        if asset_name.lower() == 'manifest': continue
        try:
            df = pd.read_csv(file_path, encoding='utf-8-sig')
            df.columns = df.columns.str.strip()

            # 🚀 원본 방식 복구: 엄격하게 date, 날짜, time 만 인정 (이상한 파일 차단)
            rename_dict = {col: 'Date' for col in df.columns if str(col).lower() in ['date', '날짜', 'time']}

            if rename_dict:
                df.rename(columns=rename_dict, inplace=True)
                raw_data_dict[asset_name] = df
        except Exception as e:
            print(f"[{asset_name}] 파일 로드 에러: {e}")

    return raw_data_dict

def preprocess_data(raw_data_dict):
    if 'vkospi' in raw_data_dict: del raw_data_dict['vkospi']
    asset_names = list(raw_data_dict.keys())
    processed_data = {}

    for name in asset_names:
        df = raw_data_dict[name].copy()

        if 'Date' not in df.columns:
            continue

        # 🚨 DateParseError("BuyHold") 에러 방어 로직
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df = df.dropna(subset=['Date'])

        df = df.sort_values('Date').set_index('Date')
        if df.empty: continue

        val_cols = [c for c in df.columns]
        target_col = val_cols[target_col_idx]

        if df[target_col].min() >= 0 and df[target_col].mean() > 1.0:
            df['Log_Return'] = np.log(df[target_col] / df[target_col].shift(1))
        else:
            df['Log_Return'] = df[target_col]

        df['rolling_mean'] = df['Log_Return'].rolling(window=rolling_window).mean()
        df['rolling_std'] = df['Log_Return'].rolling(window=rolling_window).std()
        df['scaled_value'] = (df['Log_Return'] - df['rolling_mean']) / (df['rolling_std'] + 1e-8)
        df = df.dropna(subset=['scaled_value'])

        df['target_class'] = (np.floor(df['scaled_value']).clip(-6, 5) + 6).astype(int)

        # 🌟 학습 및 검증을 위해 기간 필터링
        df = df[(df.index >= train_start) & (df.index <= val_end)]

        if not df.empty:
            processed_data[name] = df

    # 유효한 자산 목록 업데이트 (정확히 32개가 되어야 함)
    valid_asset_names = list(processed_data.keys())

    all_dates = set()
    for df in processed_data.values():
        all_dates.update(df.index)

    global_calendar = pd.Series(list(all_dates)).sort_values().reset_index(drop=True)
    return global_calendar, processed_data, valid_asset_names

# ==========================================
# 4. 동적 네트워크 사전 계산
# ==========================================
def precompute_dynamic_networks(global_calendar, processed_data, asset_names, time_window, threshold):
    print(f"⏳ 동적 네트워크 사전 계산 시작 (전체 기간)...")
    start_time = time.time()
    network_dict = {}
    bins = [-1.0, 1.0]

    for t_idx in range(time_window, len(global_calendar) - 1):
        current_date = global_calendar[t_idx]
        active_nodes = [i for i, name in enumerate(asset_names) if current_date in processed_data[name].index and not pd.isna(processed_data[name].loc[current_date, 'scaled_value'])]

        sources, targets, edge_features = [], [], []
        if active_nodes:
            for i in active_nodes:
                name_i = asset_names[i]
                current_return = processed_data[name_i].loc[current_date, 'scaled_value']
                sources.append(i); targets.append(i); edge_features.append([current_return, 1.0])
                data_i = processed_data[name_i].loc[:current_date, 'scaled_value'].tail(time_window).values

                for j in active_nodes:
                    if i == j: continue
                    name_j = asset_names[j]
                    data_j = processed_data[name_j].loc[:current_date, 'scaled_value'].tail(time_window).values

                    if len(data_i) == time_window and len(data_j) == time_window:
                        te_value = transfer_entropy(np.digitize(data_i, bins), np.digitize(data_j, bins), k=1)
                        if te_value >= threshold:
                            sources.append(i); targets.append(j); edge_features.append([current_return, te_value])

        if not sources:
            edge_index = torch.empty((2, 0), dtype=torch.long)
            edge_attr = torch.empty((0, msg_dim), dtype=torch.float)
        else:
            edge_index = torch.tensor([sources, targets], dtype=torch.long)
            edge_attr = torch.tensor(edge_features, dtype=torch.float)

        network_dict[t_idx] = (active_nodes, edge_index, edge_attr)
        if (t_idx % 500 == 0): print(f"  - {t_idx}일 완료...")

    print(f"✅ 사전 계산 완료 (소요 시간: {time.time() - start_time:.2f}초)")
    return network_dict

# ==========================================
# 5. 메인 학습 루프 (전체 데이터 통째로 학습)
# ==========================================
def main():
    my_data_dir = '/content/drive/MyDrive/Find-A final'
    raw_data_dict = load_csv_data(my_data_dir)
    if not raw_data_dict:
        print("⚠️ 로드할 수 있는 CSV 파일이 없습니다.")
        return

    global_calendar, processed_data, asset_names = preprocess_data(raw_data_dict)
    num_asset = len(asset_names)
    all_nodes_tensor = torch.arange(num_asset, dtype=torch.long).to(device)
    criterion = torch.nn.CrossEntropyLoss()

    print(f"📊 로드된 유효 전체 자산 수: {num_asset}개 (정상적으로 32개가 출력되어야 합니다)")
    if len(global_calendar) > 0:
        print(f"🗓️ 전체 학습/검증 데이터 기간: {global_calendar.min().date()} ~ {global_calendar.max().date()}\n")
    else:
        print("⚠️ 전처리 후 남은 데이터가 없습니다. 원본 데이터를 확인해주세요.")
        return

    network_cache = precompute_dynamic_networks(global_calendar, processed_data, asset_names, time_window, threshold)

    save_dir = '/content/drive/MyDrive/Find-A final'
    predict_modes = ['core', 'others', 'all']

    # 🌟 인덱스 범위 명확히 분리
    try:
        idx_train_start = global_calendar[global_calendar >= train_start].index[0]
        idx_train_start = max(time_window, idx_train_start)
        idx_train_end = global_calendar[global_calendar <= train_end].index[-1]

        idx_val_start = global_calendar[global_calendar >= val_start].index[0]
        idx_val_end = global_calendar[global_calendar <= val_end].index[-1]
    except IndexError:
        print("⚠️ 설정한 학습/검증 기간에 해당하는 데이터가 부족합니다.")
        return

    for current_mode in predict_modes:
        print("\n" + "🔥" * 40)
        print(f"🚀 [{current_mode.upper()}] 모드 Full-Batch Training 시작")
        print(f" - Train: 2010 ~ 2016 ({idx_train_start} ~ {idx_train_end})")
        print(f" - Val  : 2017 ({idx_val_start} ~ {idx_val_end})")
        print("🔥" * 40)

        seed_everything(42)
        memory = TGNMemory(num_nodes=num_asset, raw_msg_dim=msg_dim, memory_dim=memory_dim, time_dim=time_dim,
                           message_module=IdentityMessage(msg_dim, memory_dim, time_dim), aggregator_module=LastAggregator()).to(device)
        gnn = GraphEmbedding(memory_dim, memory_dim, msg_dim, time_dim).to(device)
        predictor = NodePredictor(embedding_dim=memory_dim, out_dim=num_classes).to(device)
        optimizer = torch.optim.AdamW(set(memory.parameters()) | set(gnn.parameters()) | set(predictor.parameters()), lr=0.0001, weight_decay=1e-4)

        best_val_dir_acc = 0.0

        for epoch in range(epochs):
            # --- TRAIN ---
            memory.train(); gnn.train(); predictor.train()
            memory.reset_state()
            train_loss, train_steps = 0, 0

            for t_idx in range(idx_train_start, idx_train_end):
                active_nodes, edge_index, edge_attr = network_cache.get(t_idx, ([], torch.empty((2, 0)), torch.empty((0, msg_dim))))
                if not active_nodes or edge_index.size(1) == 0: continue

                edge_index = edge_index.to(device); edge_attr = edge_attr.to(device)
                next_date = global_calendar[t_idx + 1]

                targets, valid_nodes = [], []
                for node in active_nodes:
                    name = asset_names[node]
                    is_core = name.lower() in [a.lower() for a in investable_assets]
                    is_valid = (current_mode == 'all') or (current_mode == 'core' and is_core) or (current_mode == 'others' and not is_core)

                    if is_valid and next_date in processed_data[name].index and not pd.isna(processed_data[name].loc[next_date, 'target_class']):
                        targets.append(processed_data[name].loc[next_date, 'target_class'])
                        valid_nodes.append(node)

                if not valid_nodes: continue

                y_target = torch.tensor(targets, dtype=torch.long).to(device)
                optimizer.zero_grad()

                z, last_update = memory(all_nodes_tensor)
                t_tensor = torch.full((edge_index.size(1),), t_idx, dtype=torch.long).to(device)

                h = gnn(z, last_update, edge_index, t_tensor, edge_attr)
                y_pred_logits = predictor(h[valid_nodes])

                loss = criterion(y_pred_logits, y_target)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                train_steps += 1
                memory.update_state(edge_index[0], edge_index[1], t_tensor, edge_attr)
                memory.detach()

            # --- VALIDATION ---
            memory.eval(); gnn.eval(); predictor.eval()
            val_exact, val_dir, val_total = 0, 0, 0

            with torch.no_grad():
                for t_idx in range(idx_val_start, idx_val_end):
                    active_nodes, edge_index, edge_attr = network_cache.get(t_idx, ([], torch.empty((2, 0)), torch.empty((0, msg_dim))))
                    if not active_nodes or edge_index.size(1) == 0: continue

                    edge_index = edge_index.to(device); edge_attr = edge_attr.to(device)
                    next_date = global_calendar[t_idx + 1]

                    targets, valid_nodes = [], []
                    for node in active_nodes:
                        name = asset_names[node]
                        is_core = name.lower() in [a.lower() for a in investable_assets]
                        is_valid = (current_mode == 'all') or (current_mode == 'core' and is_core) or (current_mode == 'others' and not is_core)

                        if is_valid and next_date in processed_data[name].index and not pd.isna(processed_data[name].loc[next_date, 'target_class']):
                            targets.append(processed_data[name].loc[next_date, 'target_class'])
                            valid_nodes.append(node)

                    if not valid_nodes: continue

                    y_target = torch.tensor(targets, dtype=torch.long).to(device)
                    z, last_update = memory(all_nodes_tensor)
                    t_tensor = torch.full((edge_index.size(1),), t_idx, dtype=torch.long).to(device)

                    h = gnn(z, last_update, edge_index, t_tensor, edge_attr)
                    y_pred_logits = predictor(h[valid_nodes])
                    preds = y_pred_logits.argmax(dim=1)

                    val_exact += (preds == y_target).sum().item()
                    val_dir += ((preds >= 6) == (y_target >= 6)).sum().item()
                    val_total += y_target.size(0)
                    memory.update_state(edge_index[0], edge_index[1], t_tensor, edge_attr)

            exact_acc = (val_exact / val_total) * 100 if val_total > 0 else 0
            dir_acc = (val_dir / val_total) * 100 if val_total > 0 else 0
            avg_train_loss = train_loss / train_steps if train_steps > 0 else 0

            # 최고 성능 갱신 시 드라이브에 직접 덮어쓰기 저장
            best_mark = ""
            if dir_acc > best_val_dir_acc:
                best_val_dir_acc = dir_acc
                best_mark = "🌟 (New Best & Saved!)"
                torch.save(memory.state_dict(), os.path.join(save_dir, f'best_memory_{current_mode}.pth'))
                torch.save(gnn.state_dict(), os.path.join(save_dir, f'best_gnn_{current_mode}.pth'))
                torch.save(predictor.state_dict(), os.path.join(save_dir, f'best_predictor_{current_mode}.pth'))

            if (epoch+1) % 5 == 0 or (epoch+1) == epochs:
                print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Exact: {exact_acc:.2f}% | Val Dir: {dir_acc:.2f}% {best_mark}")

    print("\n✨ 모든 모드(core, others, all)의 전체 데이터(2010~2017) 통째 학습 및 저장이 완료되었습니다! ✨")

if __name__ == "__main__":
    main()

🚀 장치 설정: cuda
📊 로드된 유효 전체 자산 수: 32개 (정상적으로 32개가 출력되어야 합니다)
🗓️ 전체 학습/검증 데이터 기간: 2010-04-01 ~ 2017-12-29

⏳ 동적 네트워크 사전 계산 시작 (전체 기간)...
  - 500일 완료...
  - 1000일 완료...
  - 1500일 완료...
  - 2000일 완료...
✅ 사전 계산 완료 (소요 시간: 281.86초)

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
🚀 [CORE] 모드 Full-Batch Training 시작
 - Train: 2010 ~ 2016 (30 ~ 1767)
 - Val  : 2017 (1768 ~ 2027)
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
Epoch 05/50 | Train Loss: 1.4573 | Val Exact: 37.07% | Val Dir: 49.41% 
Epoch 10/50 | Train Loss: 1.4520 | Val Exact: 36.84% | Val Dir: 48.94% 
Epoch 15/50 | Train Loss: 1.4468 | Val Exact: 38.02% | Val Dir: 49.76% 
Epoch 20/50 | Train Loss: 1.4430 | Val Exact: 37.72% | Val Dir: 49.65% 
Epoch 25/50 | Train Loss: 1.4381 | Val Exact: 37.72% | Val Dir: 49.70% 
Epoch 30/50 | Train Loss: 1.4320 | Val Exact: 39.02% | Val Dir: 50.53% 
Epoch 35/50 | Train Loss: 1.4296 | Val Exact: 37.78% | Val Dir: 49.59% 
Epoch 40/50 | Train Loss: 1.4216 | Val Exact: 38.37% | Val Dir: 50.41% 
Epoch 45/50 | Trai